# Análise dos erros do RF multiclasse na validação, pelo conflito entre as 7 classes

Este notebook **não treina nada**. Ele carrega o **modelo do ajuste já gravado** pelo notebook 05m (`modelo_ajuste.joblib`), refaz as previsões da **validação** e separa os erros pelas linhas **com** e **sem** conflito de rótulo **entre as 7 classes** (a tabela `prep_bal_multi_conf7`, criada pelo notebook 04mc).

Responde a três perguntas:
1. Qual o **recall de cada classe** nas linhas com conflito e nas sem conflito?
2. Que **fração dos erros** está nas linhas com conflito (comparada com o tamanho delas na validação)?
3. Nos erros das linhas com conflito, o modelo previu **a classe parceira** (a do rótulo gêmeo)?

**Antes de rodar:** os notebooks 02, 03m, 04m, 04mc e 05m precisam ter terminado. Coloque este arquivo na **mesma pasta** deles (onde está o `.env`). Só lê: não cria nem altera nada no servidor. Para analisar a versão com as colunas de AUC alto, defina `PASTA_RUN` (pasta `rf_multiclasse_com_auc\rfm_...`) e use as views `_com_auc`. Tempo: poucos minutos.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- Carga do modelo do ajuste e das previsões da validação ----------
import joblib
PASTA_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "rf_multiclasse"
PASTA_RUN = globals().get("PASTA_RUN", None)
if PASTA_RUN is None:
    runs = sorted(p for p in PASTA_BASE.glob("rfm_*") if (p / "modelo_ajuste.joblib").exists()); assert runs, "nenhuma pasta rfm_* com modelo_ajuste.joblib em " + str(PASTA_BASE) + ": rode antes o notebook 05m"
    PASTA_RUN = runs[-1]
PASTA_RUN = Path(PASTA_RUN); V_VALID, CONF_TAB = globals().get("V_VALID", "prep_vw_bal_multi_validacao"), globals().get("CONF_TAB", "prep_bal_multi_conf7")
assert existe(V_VALID) and existe(CONF_TAB), f"faltam {V_VALID} ou {CONF_TAB}: rode os notebooks 04m e 04mc"
modelo = joblib.load(PASTA_RUN / "modelo_ajuste.joblib"); CM = list(modelo.colunas_modelo_); CLASSES = json.loads((PASTA_RUN / "classes.json").read_text(encoding="utf-8"))["classes"]
va = ler_view(V_VALID); assert list(va.columns) == ["id_linha", "y", "peso"] + CM, "as colunas da view de validação não coincidem com as do modelo"
y_va = va["y"].to_numpy(int); proba_va = modelo.predict_proba(va[CM].to_numpy(float))
met = json.loads((PASTA_RUN / "metricas_validacao.json").read_text(encoding="utf-8"))["sem_pesos"]
acc = float((proba_va.argmax(axis=1) == y_va).mean())
print(f"Modelo: {PASTA_RUN.name} | {len(va):,} linhas de validação | {len(CLASSES)} classes")
print(f"{'✔' if abs(acc - met['acuracia']) < 1e-9 else '✘'} a acurácia recalculada ({acc:.5f}) coincide com a gravada pelo 05m ({met['acuracia']:.5f})")
assert abs(acc - met["acuracia"]) < 1e-9, "as previsões não reproduzem as métricas gravadas: a view de validação ou o modelo não são os do 05m"

Modelo: rfm_20261101_210334 | 59,918 linhas de validação | 7 classes
✔ a acurácia recalculada (0.91265) coincide com a gravada pelo 05m (0.91265)


In [6]:
# ---------- Erros da validação, separados pelo conflito ENTRE AS 7 CLASSES ----------
conf = consultar("SELECT toString(id_linha) AS id_linha, conflito7, conflito13, parceiros7 FROM prep_bal_multi_conf7 WHERE papel = 'validacao'")
d = pd.DataFrame({"id_linha": va["id_linha"].astype(str).values, "real": y_va, "prev": proba_va.argmax(axis=1)}).merge(conf, on="id_linha", how="left")
assert len(d) == len(va) and d["conflito7"].notna().all(), "nem todas as linhas da validação foram encontradas em prep_bal_multi_conf7"
d["conflito7"] = d["conflito7"].astype(int); d["acerto"] = (d.real == d.prev); d["classe"] = d.real.map(lambda k: CLASSES[k]); d["prev_nome"] = d.prev.map(lambda k: CLASSES[k])
t = d.groupby(["classe", "conflito7"]).agg(linhas=("acerto", "size"), recall=("acerto", "mean")).unstack("conflito7")
t.columns = [f"{a} ({'conflito entre as 7' if b == 1 else 'sem conflito'})" for a, b in t.columns]; t = t.reindex(CLASSES)
for c_ in [c for c in t.columns if c.startswith("linhas")]: t[c_] = t[c_].fillna(0).astype(int)
print("== Recall da validação, nas linhas COM e SEM conflito entre as 7 classes ==\n"); display(t.round(4))
tot = d.groupby("conflito7").agg(linhas=("acerto", "size"), acuracia=("acerto", "mean"), erros=("acerto", lambda s: int((~s).sum()))); print("\nGeral:"); display(tot.round(4))
err = d[~d.acerto]; e1 = err[err.conflito7 == 1]
no_parceiro = sum(p in par.split("|") for p, par in zip(e1.prev_nome, e1.parceiros7))
print(f"Dos {len(err):,} erros, {len(e1):,} ({100 * len(e1) / len(err):.1f}%) estão em linhas com conflito entre as 7 classes (que são {100 * (d.conflito7 == 1).mean():.1f}% da validação).")
print(f"Nesses {len(e1):,} erros, o modelo previu justamente a classe parceira (a do rótulo gêmeo) em {no_parceiro:,} ({100 * no_parceiro / max(len(e1), 1):.1f}%).")
print(f"Taxa de erro: {100 * len(e1) / max(int((d.conflito7 == 1).sum()), 1):.1f}% nas linhas com conflito × {100 * (len(err) - len(e1)) / max(int((d.conflito7 == 0).sum()), 1):.1f}% nas sem conflito.")

== Recall da validação, nas linhas COM e SEM conflito entre as 7 classes ==



,linhas (sem conflito),linhas (conflito entre as 7),recall (sem conflito),recall (conflito entre as 7)
classe,,,,
BENIGN,9998,2,0.9992,0.0000
LDAP,5662,358,0.9832,0.6704
MSSQL,9409,591,0.9296,0.0964
NetBIOS,2186,1712,0.9209,0.5613
Syn,9862,138,0.9653,0.2174
UDP,9906,94,0.9254,0.1383
UDPLag,8927,1073,0.8742,0.5359



Geral:


,linhas,acuracia,erros
conflito7,,,
0,55950,0.9438,3142
1,3968,0.4728,2092


Dos 5,234 erros, 2,092 (40.0%) estão em linhas com conflito entre as 7 classes (que são 6.6% da validação).
Nesses 2,092 erros, o modelo previu justamente a classe parceira (a do rótulo gêmeo) em 2,015 (96.3%).
Taxa de erro: 52.7% nas linhas com conflito × 5.6% nas sem conflito.
